# Run the transfer matrix

Drives `src/train.py` over one owner's half of the matrix: 2 strategies x 7 source
generators x 1 seed = **14 runs per pass**, 28 per person across the two required seeds,
56 in total. Each run trains a CompactCNN from scratch on one generator, evaluates it
against all seven, and writes `results/runs/<strategy>/<source>/seed<k>/metrics.json`
with the selected weights as `model.pt` beside it.

Run the cells top to bottom. Two of them are gates, and neither is optional:

- **The cache check** - a missing or half-written shard is twenty seconds to spot here and
  fourteen wasted GPU-minutes to spot inside the loop.
- **The smoke run, then the calibration run** - no forward or backward pass in this project
  has ever executed anywhere. The smoke run costs one arm load and two epochs; finding a
  broken loop on run 9 of 14 costs the afternoon.

The whole resume mechanism is `if metrics.json exists: skip`, so a Colab disconnect costs at
most the run that was in flight. Re-run the batch cell and it picks up where it stopped.

## Setup

Set `OWNER` in the next cell and nothing else: it selects the two strategies this account
trains. Both accounts read the cache from the same path - it is built in Ido's My Drive and
reached from Noa's through a shortcut placed at the matching path - so there is one `DRIVE`
for both.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob, time, shutil, subprocess
from pathlib import Path

# Confirm a GPU is actually attached before anything else. Free-tier Colab will happily
# hand out a CPU runtime, and the batch would then take days rather than two hours.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

from google.colab import drive
drive.mount('/content/drive')

# Who is running this notebook. It picks the two strategies trained here, which is how the
# two Google accounts avoid duplicating a run: Ido owns centre_crop and rescale, Noa owns
# pad and random_crop (PLAN.md, "Who owns what").
OWNER = "ido"
OWNED = {"ido": ("centre_crop", "rescale"), "noa": ("pad", "random_crop")}
MY_STRATEGIES = OWNED[OWNER]

# One path for both accounts. The cache is built once, in Ido's My Drive; Noa reaches the
# same bytes through a shortcut she placed at the matching path, so nothing is copied and
# nothing is rebuilt - both accounts read the identical shards.
DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

assert os.path.isdir(DRIVE), (
    f"not found: {DRIVE}\nCheck it is mounted on THIS account: `ls /content/drive/MyDrive`."
)

# Reads and writes are deliberately not the same root. The cache is read from wherever it is
# shared to this account; the backup is written to this account's OWN My Drive, which always
# exists and is always writable. A view-only share would otherwise fail the copytree at the
# very end, after the batch had already run.
BACKUP = f"/content/drive/MyDrive/deep_learning_results/{OWNER}"

# Repo-relative on purpose: metrics.json is the sync channel between the two accounts and
# has to sit inside the git tree to be committed. The weights land in the same directories
# and are gitignored (`*.pt`); the last cell copies those to Drive, which is what survives
# a runtime recycle.
RESULTS = "results/runs"

print(f"owner {OWNER} -> strategies {MY_STRATEGIES}")
print("cache <-", CACHE, "" if os.path.isdir(CACHE) else "  <- MISSING, see the gate below")
print("backup ->", BACKUP)

## Get the code

In [ ]:
# Idempotent: clones on the first run, pulls on every later one.
![ -d /content/deep-learning ] || git clone -q https://github.com/noa-keter/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git pull --ff-only
!git log --oneline -1

# requirements.txt is deliberately unpinned, so this resolves against the torch Colab
# already ships and installs nothing. Never pin torch in it - a pinned version would
# replace Colab's CUDA build with a multi-gigabyte download.
!pip -q install -r requirements.txt

# Explicit, so `import src...` below does not depend on how the kernel resolves a bare
# cwd entry in sys.path.
REPO = os.getcwd()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from src.data import GENERATORS, load_meta
from src.train import metrics_path

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print(f"torch {torch.__version__}  cuda {torch.cuda.is_available()}  {gpu_name}")
assert torch.cuda.is_available(), "no GPU on this runtime - Runtime > Change runtime type > T4"
DEVICE = "cuda"

## Does the cache actually exist?

`load_arm_numpy` reads metadata for **both** splits - the train split it samples the arm
from, and the split the dataset calls `validation`, which is our test set. A
validation-only build is therefore not enough to train a single run.

Two checks. Exact byte sizes per shard, which catches anything that stopped mid-write, and
the row composition per split, which catches a build that finished the wrong thing.

In [ ]:
import numpy as np

# Exact expected bytes: rows * 128*128*3, plus the 128-byte .npy header. Only the two
# strategies this account trains have to be present; all four are reported anyway, because
# the other pair being absent means the partner's half cannot run from this cache.
for strategy in ["centre_crop", "random_crop", "rescale", "pad"]:
    for split, n_shards, per_shard in [("validation", 4, 1750), ("train", 14, 2000)]:
        files = sorted(glob.glob(f"{CACHE}/{strategy}/{split}-*.npy"))
        sizes = sorted({os.path.getsize(f) for f in files})
        ok = len(files) == n_shards and sizes == [per_shard * 128 * 128 * 3 + 128]
        mine = "*" if strategy in MY_STRATEGIES else " "
        print(f"{mine}{'OK ' if ok else 'BAD'} {strategy:<12}{split:<11} {len(files)}/{n_shards} files  {sizes}")

# Composition, from the metadata alone - no pixels are read. Real is 3,500 / 14,000 and
# every generator is exactly 500 / 2,000. SD14 is a declared class name with zero rows and
# must never appear; the seven-generator claim in the report depends on it.
for split, expected_real in [("validation", 3500), ("train", 14000)]:
    meta = load_meta(CACHE, split)
    tags, counts = np.unique(meta["generator"], return_counts=True)
    by_tag = dict(zip(tags.tolist(), counts.tolist()))
    print(f"\n{split}: {len(meta['generator'])} rows")
    print("  " + "  ".join(f"{t}:{c}" for t, c in by_tag.items()))
    missing = set(GENERATORS) - set(by_tag)
    assert not missing, f"{split} is missing generators: {missing}"
    assert "SD14" not in by_tag, "SD14 has rows - the seven-generator claim would be wrong"
    # Counted as the remainder rather than by tag name, so the check does not depend on how
    # the dataset spells its real class.
    n_real = len(meta["generator"]) - sum(by_tag[g] for g in GENERATORS)
    assert n_real == expected_real, f"{split}: {n_real} real rows, expected {expected_real}"

print("\ncache OK for both splits")

## Smoke run: prove the loop executes at all

Two epochs on 200 images per class, written to a throwaway results directory so it can
never be mistaken for a real cell. It exercises everything the batch depends on - the arm
load, AMP, the flip, the cosine schedule, checkpoint selection, the seven-cell scoring,
the checkpoint and JSON writes - in a couple of minutes.

The architecture check above it needs no data and no cache, so it goes first: if the
parameter count or the gradient path is wrong, nothing else is worth trying.

In [ ]:
SMOKE = "/content/smoke_runs"
!rm -rf {SMOKE}

# Shapes, the 1,173,473-parameter count, and an overfit-ten-examples test that would fail
# on a detached tensor or a dead ReLU stack. Under a minute, and it needs no cache at all.
!python src/model.py

# The first real training run this project has ever executed. Deliberately tiny in epochs
# and training rows - but the test set is always the full 4,000, so `load_s` here is the
# honest read cost of one arm off Drive.
smoke_cmd = (
    f'python -m src.train --cache-dir "{CACHE}" --strategy {MY_STRATEGIES[0]}'
    f' --source BigGAN --seed 0 --epochs 2 --train-per-class 200'
    f' --results-dir "{SMOKE}" --device {DEVICE}'
)
!{smoke_cmd}

smoke = json.loads(Path(f"{SMOKE}/{MY_STRATEGIES[0]}/BigGAN/seed0/metrics.json").read_text())
print(f"\ncells        {len(smoke['cells'])} (expect 7), sizes {sorted(set(smoke['cell_sizes'].values()))} (expect [1000])")
print(f"load_s       {smoke['load_s']:.0f}s of {smoke['wall_clock_s']:.0f}s wall clock")
print(f"peak vram    {smoke['peak_vram_mb']:.0f} MB")
print(f"checkpoint   {smoke['checkpoint']}  ({os.path.getsize(smoke['checkpoint']) / 1e6:.1f} MB)")
print("\nthe loop executes end to end - accuracy at two epochs means nothing, ignore it")

## Calibration: one real run, against the timing gate

A full 40-epoch run on `MY_STRATEGIES[0]` / BigGAN / seed 0. It is a real cell and counts
as one of this account's fourteen - nothing here is thrown away.

BigGAN on purpose: it is native 128x128, so all four strategies are the identity on it and
its whole matrix row is the study's control.

| Time for this run | Then |
|---|---|
| ≤ 4 min | proceed with 40 epochs |
| 4–8 min | set `EPOCHS = 25` in the batch cell |
| > 8 min | `EPOCHS = 25` **and** `TRAIN_PER_CLASS = 1000` |

**In-domain accuracy must clear 95 %.** That is what says the pipeline learns, before
thirteen more runs are committed to it. If it does not clear it, stop - go back to the
pilot contact sheet from notebook 00 rather than tuning anything here.

In [ ]:
calib_cmd = (
    f'python -m src.train --cache-dir "{CACHE}" --strategy {MY_STRATEGIES[0]}'
    f' --source BigGAN --seed 0 --results-dir "{RESULTS}" --device {DEVICE}'
)
!{calib_cmd}

In [ ]:
calib = json.loads(metrics_path(RESULTS, MY_STRATEGIES[0], "BigGAN", 0).read_text())
minutes = calib["wall_clock_s"] / 60
verdict = "OK - clears 95%" if calib["in_domain"] > 0.95 else "STOP - under 95%, do not start the batch"

print(f"in-domain      {calib['in_domain']:.4f}   {verdict}")
print(f"off-domain     {calib['off_domain_mean']:.4f}   (transfer - no gate on this, it is the result)")
print(f"best epoch     {calib['best_epoch'] + 1} of {calib['config']['epochs']}")
print(f"wall clock     {minutes:.1f} min, of which load {calib['load_s'] / 60:.1f} min")

if minutes <= 4:
    print("\ngate -> keep EPOCHS = 40, TRAIN_PER_CLASS = 2000")
elif minutes <= 8:
    print("\ngate -> set EPOCHS = 25 in the next cell")
else:
    print("\ngate -> set EPOCHS = 25 and TRAIN_PER_CLASS = 1000 in the next cell")

# Drive FUSE reads are the one cost that does not shrink when the epochs do. If the load is
# a large share of the run, the local-disk copy further down pays for itself over 14 runs.
if calib["load_s"] > 0.3 * calib["wall_clock_s"]:
    print("load is >30% of the run -> run the 'copy the cache to local disk' cell before the batch")

## The batch - fourteen runs

Seed-major and resumable. Each run is a fresh `python -m src.train` subprocess, which costs
a few seconds of CUDA init and buys two things worth more than that: VRAM is fully released
between runs, and a failure in one run leaves the loop alive to finish the other thirteen.

Seed 0 first. **Seed 1 is required, not a bonus** - the headline claim is a change in the
generator ranking, and one seed cannot carry that against the ±1.6 pp binomial SE per
cell. Once seed 0 is complete, set `SEEDS = [1]` and re-run this cell, leaving `EPOCHS` and
`TRAIN_PER_CLASS` exactly as they were for seed 0.

In [ ]:
SEEDS = [0]              # then [1] once seed 0 is complete. Two seeds is the floor.
EPOCHS = 40              # lower it only if the calibration gate said so
TRAIN_PER_CLASS = 2000   # halve it only if the calibration gate said so

queue = [(st, src, sd) for sd in SEEDS for st in MY_STRATEGIES for src in GENERATORS]
print(f"{len(queue)} runs queued: {MY_STRATEGIES} x {len(GENERATORS)} generators x seeds {SEEDS}")
print(f"{EPOCHS} epochs, {TRAIN_PER_CLASS} train images per class\n")

elapsed_s, failures = [], []
for index, (strategy, source, seed) in enumerate(queue, start=1):
    label = f"{strategy}/{source}/seed{seed}"
    out = metrics_path(RESULTS, strategy, source, seed)
    # The entire resume mechanism. metrics.json is written last, after the weights, so a
    # run directory that has it has everything.
    if out.exists():
        print(f"[{index:>2}/{len(queue)}] skip {label}")
        continue

    print(f"[{index:>2}/{len(queue)}] {label}", flush=True)
    started = time.perf_counter()
    proc = subprocess.run([
        sys.executable, "-m", "src.train",
        "--cache-dir", CACHE,
        "--strategy", strategy,
        "--source", source,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--train-per-class", str(TRAIN_PER_CLASS),
        "--results-dir", RESULTS,
        "--device", DEVICE,
    ])
    took_s = time.perf_counter() - started

    if proc.returncode != 0 or not out.exists():
        failures.append(label)
        print(f"           FAILED (exit {proc.returncode}) after {took_s / 60:.1f} min - continuing\n")
        continue

    elapsed_s.append(took_s)
    run = json.loads(out.read_text())
    eta_min = (len(queue) - index) * (sum(elapsed_s) / len(elapsed_s)) / 60
    print(
        f"           {took_s / 60:.1f} min "
        f"(load {run['load_s'] / 60:.1f} min, {100 * run['load_s'] / run['wall_clock_s']:.0f}% of it)  "
        f"in-domain {run['in_domain']:.3f}  off-domain {run['off_domain_mean']:.3f}  "
        f"~{eta_min:.0f} min left\n",
        flush=True,
    )

skipped = len(queue) - len(elapsed_s) - len(failures)
print(f"done: {len(elapsed_s)} run, {skipped} already present, {len(failures)} failed")
if failures:
    print("failed - re-run this cell to retry them:", failures)

## If `load_s` dominates: copy the cache to local disk

Drive is a FUSE mount, and every run reads ~390 MB of training images plus the full
4,000-image test set through it. Over fourteen runs that read is paid fourteen times.

Copying the two strategies this account needs into `/content` is ~3.5 GB and a few minutes
against Colab's ~100 GB of local disk, and every later read is then local. Worth it if the
loop reports load as a large share of each run, pointless otherwise - and it does **not**
survive a runtime recycle, so it has to be redone after a disconnect.

The cell rebinds `CACHE`, so re-running the batch cell afterwards picks up the local copy
with no other change.

In [ ]:
LOCAL_CACHE = "/content/cache"

os.makedirs(LOCAL_CACHE, exist_ok=True)
# meta/ is small and needed for both splits; only the owned strategies carry pixels worth
# copying. dirs_exist_ok makes a re-run cheap instead of an error.
for sub in ("meta", *MY_STRATEGIES):
    started = time.perf_counter()
    shutil.copytree(f"{CACHE}/{sub}", f"{LOCAL_CACHE}/{sub}", dirs_exist_ok=True)
    copied = Path(f"{LOCAL_CACHE}/{sub}").rglob("*")
    gb = sum(f.stat().st_size for f in copied if f.is_file()) / 1e9
    print(f"{sub:<12} {gb:.2f} GB in {time.perf_counter() - started:.0f}s")

CACHE = LOCAL_CACHE
print("\ncache <-", CACHE, "(local disk - lost on a runtime recycle, redo this cell then)")

## Hand off the results

Two channels, for two different reasons.

- **`metrics.json` goes through git.** About 2 KB per run, and it is what merges the two
  accounts' halves of the matrix into one results tree. No large transfer between accounts,
  ever.
- **`model.pt` goes to Drive.** `*.pt` is gitignored, and `/content` is wiped when the
  runtime recycles - but the input-gradient attribution figure needs one trained model per
  strategy, so losing the weights means re-running the matrix to get them back.

In [ ]:
# Weights and metrics both, into this account's own My Drive (see BACKUP in the setup cell)
# so the two accounts cannot overwrite each other's copy and neither depends on write access
# to the other's folder.
backup = f"{BACKUP}/runs"
os.makedirs(backup, exist_ok=True)
shutil.copytree(RESULTS, backup, dirs_exist_ok=True)
n_metrics = len(glob.glob(f"{backup}/**/metrics.json", recursive=True))
n_ckpt = len(glob.glob(f"{backup}/**/model.pt", recursive=True))
print(f"{n_metrics} metrics.json and {n_ckpt} checkpoints -> {backup}")

# Only the metrics reach the commit; .gitignore keeps the weights out on its own.
!git add {RESULTS}
!git -c user.name="{OWNER}" -c user.email="{OWNER}@colab" commit -q -m "Add {OWNER}'s matrix runs" || echo "nothing new to commit"
!git status --short {RESULTS}
!git log --oneline -1

print("\nPush from wherever git credentials already live - a Colab runtime is not the place")
print("to put a token:  git pull --rebase  then  git push")